# Secondary Documentation Notebook

This notebook collects visualizations, debug snippets, and analyses that are **not** part of
the main processing pipeline. It is meant to be run *after* `main_notebook.ipynb` has completed,
since it reads intermediate data saved by that notebook.

**Contents:**
1. Setup & Data Loading
2. FFT Fingerprint Visualization (single-tile debug)
3. FFT Effects on Sample Images
4. FFT Amplitude Profiles per Cluster
5. Downloading DEMs for Representative Areas
6. Extracting Sample Images for Documentation
7. Reprojecting Representative Samples
8. Geographic Descriptions of Representative Positions

**Prerequisites:** Run `main_notebook.ipynb` first so that the following files exist:
- `output/tabular_and_text/augmented_dems_doc_data.pkl`
- `output/tabular_and_text/representative_positions.pkl`

## Setup & Data Loading

Loads the pickled DEM data from the main pipeline and reconstructs a `temporary_data` dict
with `SimpleNamespace` objects, so the original code cells work without modification.

In [ ]:
import pickle
import types
import numpy as np
import os

# Zwischendaten aus dem Haupt-Notebook laden
with open("output/tabular_and_text/augmented_dems_doc_data.pkl", "rb") as f:
    doc_data = pickle.load(f)

# temporary_data-Struktur aufbauen, damit alter Code unverändert läuft
temporary_data = {"augmented_dems": []}
for dem_dict in doc_data["augmented_dems"]:
    ns = types.SimpleNamespace(**dem_dict)
    temporary_data["augmented_dems"].append(ns)

print(f"Loaded {len(temporary_data['augmented_dems'])} DEM(s).")

## FFT Fingerprint Visualization

Inspects the FFT magnitude vector of a single tile (at the midpoint of the first DEM).
Each bar represents one concentric ring level of the FFT spectrum — from low spatial
frequencies (inner rings) to high frequencies (outer rings).
The `wandering_mask` isolates one level at a time to show its individual contribution.

In [ ]:
# Einen Stichproben-Index in der Mitte der Daten auswählen
sample_index = len(temporary_data["augmented_dems"][0].fft_magnitude_all_levels)//2
temporary_data["augmented_dems"][0].fft_magnitude_all_levels[sample_index]

In [ ]:
print(temporary_data["augmented_dems"][0].tile_centers_orig[sample_index].y)
print(temporary_data["augmented_dems"][0].tile_centers_orig[sample_index].x)

In [ ]:
import matplotlib.pyplot as plt


median_amplitudes = np.reshape(temporary_data["augmented_dems"][0].fft_magnitude_all_levels[sample_index], (1,-1))
number_of_examples = 1
number_of_samples = 17
ncols = 3
nrows = 10
median_amplitudes.shape

In [ ]:
wandering_mask = []
length = 17
for i in range(17):
    new_line = np.array(range(length)) == i
    wandering_mask.append(new_line.astype(int))

In [ ]:

shown_amplitudes = median_amplitudes
fig, ax = plt.subplots(figsize=(10, 2)
)

vmin, vmax = (median_amplitudes.min(), median_amplitudes.max())

heights = shown_amplitudes[0]
colors = plt.cm.viridis(1-((heights - vmin) / (vmax - vmin)))


ax.bar(np.array(range(number_of_samples))+1, shown_amplitudes[0], color=colors)
ax.set_axisbelow(True)
ax.grid(axis="x", visible=True, alpha=0.2, linestyle="--", which="major")
ax.grid(axis="y", visible=True, alpha=0.9, linestyle="--", which="major")
ax.set_yticks([0, 2, 4, 6, 8, 10])
ax.set_yticklabels(["0","","4","","","10"])
ax.set_xticks(np.array(range(number_of_samples + 1)) - 0.5)
ax.set_xticklabels([],)

fig.tight_layout()
plt.show()

In [ ]:
for l in range(length):

    shown_amplitudes = median_amplitudes * wandering_mask[l]
    fig, ax = plt.subplots(figsize=(10, 2)
    )

    vmin, vmax = (median_amplitudes.min(), median_amplitudes.max())

    heights = shown_amplitudes[0]
    colors = plt.cm.viridis(1-((heights - vmin) / (vmax - vmin)))


    ax.bar(np.array(range(number_of_samples))+1, shown_amplitudes[0], color=colors)
    ax.set_axisbelow(True)
    ax.grid(axis="x", visible=True, alpha=0.2, linestyle="--", which="major")
    ax.grid(axis="y", visible=True, alpha=0.9, linestyle="--", which="major")
    ax.set_yticks([0, 2, 4, 6, 8, 10])
    ax.set_yticklabels(["0","","4","","","10"])
    ax.set_xticks(np.array(range(number_of_samples + 1)) - 0.5)
    ax.set_xticklabels([],)

    fig.tight_layout()
    plt.show()

In [ ]:
import pickle
import io

# Figure in Buffer serialisieren
buf = io.BytesIO()
pickle.dump(fig, buf)

for i in range(30):
    # Figure aus Buffer kopieren
    buf.seek(0)
    fig_copy = pickle.load(buf)
    fig_copy.set_size_inches(10 * 3, 15 * 3)

    for ax_x in fig_copy.axes:
        ax_x.set_yticklabels([])
        ax_x.tick_params(left=False, bottom=False)
        ax_x.set_xlim([0.5, 17.5])
        ax_x.set_title("")
        for spine in ax_x.spines.values():
            spine.set_edgecolor("grey")
            spine.set_linewidth(0.5)

    for j, axis in enumerate(fig_copy.axes):
        if j != i:
            fig_copy.delaxes(axis)

    row = i // 3
    col = i % 3

    row += 1
    col += 1

    fig_copy.savefig(
        f"output/sample_fft_{row:02d}_{col:02d}.png", bbox_inches="tight", dpi=200
    )
    plt.close(fig_copy)

## FFT Effects on Sample Images

This section illustrates what the FFT decomposition does to a landscape image.
It was originally developed as a standalone notebook (`documentation_fft_effects.ipynb`).

# Showing what FFT does
This notebook is supposed to illustrate the effects of FFT on landscapes.
It does that by showing which features persist when selecting a certain frequency range from a landscape tile’s FFT magnitude.

In [ ]:
import numpy as np
import pyfftw
import cv2
import matplotlib.pyplot as plt
import common
from sklearn.preprocessing import MinMaxScaler
import subprocess
import ffmpeg
import numpy as np
from scipy.ndimage import gaussian_filter

In [ ]:
NUMBER_OF_SAMPLES = 17
SIDE_LENGTH_FFT = 23

In [ ]:
def gaussian_filter_weighted(input: np.ndarray, sigma):
    # Gültigkeitsmaske: überall 1
    valid = np.ones_like(input)

    num = gaussian_filter(input, sigma=sigma, mode="constant", cval=0)
    den = gaussian_filter(valid, sigma=sigma, mode="constant", cval=0)

    ergebnis = num / den

    return ergebnis

In [ ]:
def normalize(array):
    factor = np.max(array) - np.min(array)
    if factor == 0: return np.zeros((1))
    return (array - np.min(array)) / factor

In [ ]:
image_source_path = "/Users/scharnagl/Documents/GitHub/geospatial-landscape-clustering-by-fft/documentation/Representative_Samples/16_bit_heightmaps/schottland_ben_nevis_2_aeqd_-4.9750_56.7600_ps.tif"

image_source_path = "/Users/scharnagl/Documents/GitHub/geospatial-landscape-clustering-by-fft/documentation/Representative_Samples/16_bit_heightmaps/italy_sample_tile_aeqd_11.2955_42.9107.tif"

image_pixels = cv2.imread(image_source_path, cv2.IMREAD_ANYDEPTH)
#image_pixels = (image_pixels - np.min(image_pixels)) / (np.max(image_pixels) - np.min(image_pixels))

# CAUTION resize

plt.imshow (image_pixels)

In [ ]:
circle_masks_old = common.RingImageSeries(*image_pixels.shape,NUMBER_OF_SAMPLES, 1)
circle_masks_log = common.RingImageSeriesLog(*image_pixels.shape, NUMBER_OF_SAMPLES, 1)
circle_masks = common.RingImageSeriesLogFine(*image_pixels.shape, steps = NUMBER_OF_SAMPLES, animation_steps=NUMBER_OF_SAMPLES*10, bandwidth=1)

In [ ]:
def generate_fft_sample_data(image, circle_masks, side_px=23, steps=17, hanning=False):

    use_image = cv2.resize(
        image, (side_px, side_px), interpolation=cv2.INTER_CUBIC
    )

    fft_input_array = pyfftw.empty_aligned(
        (1, use_image.shape[0], use_image.shape[1]), dtype="complex64"
    )

    fft_output_array = pyfftw.empty_aligned(
        (1, use_image.shape[0], use_image.shape[1]), dtype="complex64"
    )

    fft_execution_plan = pyfftw.FFTW(
        fft_input_array,
        fft_output_array,
        axes=(1, 2),
        direction="FFTW_FORWARD",
        flags=("FFTW_MEASURE",),
    )

    if hanning:
        hanning_x = np.hanning(use_image.shape[0])
        hanning_y = np.hanning(use_image.shape[1])
        hanning_2d = np.outer(hanning_x, hanning_y)

        fft_input_array[:] = (
            use_image - np.mean(use_image, keepdims=True)
        ) * hanning_2d
    else:
        fft_input_array[:] = use_image - np.mean(use_image, keepdims=True)

    fft_execution_plan.execute()

    fft_output_abs = np.log(np.abs(fft_output_array) + 1)
    fft_output_abs_shifted = np.fft.fftshift(fft_output_abs, axes=(1, 2))

    fft_output_abs_shifted_repeated = np.repeat(
        fft_output_abs_shifted, len(circle_masks), axis=0
    )
    fft_output_repeated_masked = fft_output_abs_shifted_repeated * circle_masks

    fft_fingerprint = np.divide(
        np.sum(fft_output_repeated_masked, axis=(1, 2)), np.sum(circle_masks, axis=(1, 2))
    )

    #plt.imshow(use_image)
    #plt.show()
    #print(fft_output_abs_shifted.shape)
    #plt.imshow(fft_output_abs_shifted[0])
    #plt.show()
    #plt.bar(x=range(len(fft_fingerprint)), height=fft_fingerprint)
    #plt.show()

    return fft_fingerprint

In [ ]:
from time import perf_counter_ns

In [ ]:
NUMBER_OF_SAMPLES = 17
all_results = []
all_resolutions = []
all_times = []

for i in range(23,1024,3):
    start = perf_counter_ns()
    # Hyperparameter: ring image series, hanning, side_px
    SIDE_LENGTH_FFT = i
    all_resolutions.append(SIDE_LENGTH_FFT)
    use_masks = common.RingImageSeriesLog(*(SIDE_LENGTH_FFT,SIDE_LENGTH_FFT),NUMBER_OF_SAMPLES, 1)
    result = generate_fft_sample_data(image_pixels, use_masks, side_px=SIDE_LENGTH_FFT, steps=17, hanning=True)
    all_results.append(result)
    all_times.append(perf_counter_ns() - start)


In [ ]:

all_results = np.array(all_results)
all_resolutions = np.array(all_resolutions)
all_times = np.array(all_times)
delta_resuls = all_results - all_results[-1]
delta_resuls_len = np.linalg.norm(delta_resuls, axis=1)

In [ ]:
all_resolutions.shape

In [ ]:
all_times.shape

In [ ]:
y.shape

In [ ]:
np.mean(x_multi_poly)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

y = all_times.reshape(-1,1)
x = all_resolutions.reshape(-1,1)
x_multi = np.hstack((
    np.log(x),
    x,
    1/x

))

poly_transform = PolynomialFeatures(8)
x_multi_poly = poly_transform.fit_transform(x_multi)

regressor = LinearRegression(n_jobs=-1)
regressor.fit(x_multi_poly, y)

yhat = regressor.predict(x_multi_poly)

plt.scatter(all_times, all_resolutions, c="blue")
plt.scatter(all_times, yhat.ravel(), c="red")

In [ ]:
test_x = np.linspace(1,500)
test_y = np.log(test_x)

plt.scatter(test_x, test_y)

In [ ]:
from kneed import KneeLocator
kneedle = KneeLocator(
    all_times,
    yhat.ravel(),
    curve="convex",
    direction="decreasing"
)

print("Optimal Time", kneedle.knee)

In [ ]:
index_of_resolution = np.where(all_times == kneedle.knee)[0]
print("Optimal Resolution", all_resolutions[index_of_resolution])

In [ ]:
SIDE_LENGTH_FFT = 73
use_masks = common.RingImageSeriesLog(*(SIDE_LENGTH_FFT,SIDE_LENGTH_FFT),NUMBER_OF_SAMPLES, 1)


In [ ]:
%%timeit
result = generate_fft_sample_data(image_pixels, use_masks, side_px=SIDE_LENGTH_FFT, steps=17, hanning=True)

In [ ]:
# i = 140 ist stabil, ohne hanning
# i = 125 ist stabil, mit hanning
# ab i = 220 sehr stabil (mit hanning, ohne log)
# ab i = 70 stabil (mit hanning, mit log), ab 150 sehr stabil
# kneedle sagt: 254

In [ ]:
wandering_mask = np.zeros((NUMBER_OF_SAMPLES*3,NUMBER_OF_SAMPLES))

for i in range(NUMBER_OF_SAMPLES):
    wandering_mask[i,i] = 1
    wandering_mask[(NUMBER_OF_SAMPLES*2)-i-1,i:] = 1
    wandering_mask[(NUMBER_OF_SAMPLES*2)+i,:NUMBER_OF_SAMPLES-i] = 1

wandering_mask = np.flip(wandering_mask, axis=1)

for l in range(len(wandering_mask)):

    shown_amplitudes = fft_fingerprint
    fig, ax = plt.subplots(figsize=(10, 2)
    )

    vmin, vmax = (fft_fingerprint.min(), fft_fingerprint.max())

    heights = shown_amplitudes[0]
    colors = plt.cm.viridis(1-((heights - vmin) / (vmax - vmin)))
    opacity = np.clip(wandering_mask[l] + 0.15, 0, 1)

    barcontainer = ax.bar(np.array(range(NUMBER_OF_SAMPLES))+1, shown_amplitudes, color=colors)
    for n, bar in enumerate(barcontainer):
        bar.set_alpha(opacity[n])


    ax.set_axisbelow(True)
    ax.grid(axis="x", visible=True, alpha=0.2, linestyle="--", which="major")
    ax.grid(axis="y", visible=True, alpha=0.9, linestyle="--", which="major")
    ax.set_yticks([0, 2, 4, 6, 8, 10])
    ax.set_yticklabels(["0","","4","","","10"])
    ax.set_xticks(np.array(range(NUMBER_OF_SAMPLES + 1)) - 0.5)
    ax.set_xticklabels([],)

    ax.set_yticklabels([])
    ax.tick_params(left=False, bottom=False)
    ax.set_xlim([0.5, 0.5 + NUMBER_OF_SAMPLES])   


    fig.tight_layout()
    # plt.savefig(f"documentation/fft_magnitudes_animation/fft_pingpong_steps_{l:02d}.png")
    plt.show()

In [ ]:
fft_shifted = np.fft.fftshift(fft_output_array, axes=(1,2))
fft_magnitude_display = np.log(np.abs(fft_shifted) + 1)
plt.imshow(fft_magnitude_display[0])

In [ ]:
result_images = []

In [ ]:
for circle_mask in circle_masks:


    fft_filtered = fft_shifted * circle_mask[np.newaxis, :, :]
    fft_filtered_unshifted = np.fft.ifftshift(fft_filtered, axes=(1,2))
    ifft_input_array = pyfftw.empty_aligned(fft_output_array.shape, dtype="complex64")
    ifft_output_array = pyfftw.empty_aligned(fft_output_array.shape, dtype="complex64")

    ifft_execution_plan = pyfftw.FFTW(
        ifft_input_array,
        ifft_output_array,
        axes=(1,2),
        direction = "FFTW_BACKWARD",
        flags=("FFTW_MEASURE",)
    )

    ifft_input_array[:] = fft_filtered_unshifted
    ifft_execution_plan.execute()

    N = image_pixels.shape[0] * image_pixels.shape[1] 
    result = np.real(ifft_output_array) / N
    result_images.append(np.squeeze(result, axis=0))

In [ ]:
result_images_np = np.nan_to_num(np.array(result_images),nan=0)
result_images_np.shape

In [ ]:
def write_ffv1(frames: list[np.ndarray], output: str, fps: int = 24):
    h, w = frames[0].shape
    cmd = [
        "ffmpeg", "-y",
        "-f", "rawvideo",
        "-pix_fmt", "rgb48le",
        "-s", f"{w}x{h}",
        "-r", str(fps),
        "-i", "pipe:",
        "-vcodec", "ffv1",
        "-pix_fmt", "rgb48le",
        "-color_range", "pc",
        "-color_trc", "linear",
        "-colorspace", "rgb",
        "-color_primaries", "bt709",
        output
    ]
    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE)
    for frame in frames:
        frame_rgb = np.stack([frame, frame, frame], axis=-1)
        proc.stdin.write(frame_rgb.astype("<u2").tobytes())
    proc.stdin.close()
    proc.wait()

In [ ]:


result_images_std = np.std(result_images_np, axis=(1,2), keepdims=True)
result_images_mean = np.mean(result_images_np, axis=(1,2), keepdims=True)

result_images_std_filtered = gaussian_filter(result_images_std, sigma=5)
result_images_mean_filtered = gaussian_filter(result_images_mean, sigma=10)

In [ ]:

result_images_normalized = result_images_np - result_images_mean_filtered

min_result = np.min(result_images_normalized)
max_result = np.max(result_images_normalized)

result_images_normalized -= min_result
result_images_normalized /= max_result - min_result

In [ ]:
def to_srgb(data):
    frame_srgb = np.where(data <= 0.0031308,
    12.92 * data,
    1.055 * np.power(data, 1/2.4) - 0.055)

    return frame_srgb

out_anim = to_srgb(result_images_normalized)
#out_anim = np.zeros(shape=(170,300,300))
#out_anim += to_srgb(0.5)

In [ ]:

out_anim_uint16 = (np.nan_to_num(out_anim[1:],nan=0) * 255 * 255).astype(np.uint16)

write_ffv1(out_anim_uint16,"output/fft_bandpass/fft_anim_single_italy_reverse.mkv", fps=25)



In [ ]:
def geschummerte_darstellung(array):
    shifted = np.roll(array,1,axis=0)
    shifted = np.roll(shifted,1,axis=1)
    visuals = array - shifted
    plt.imshow(visuals, cmap="grey", vmax=10, vmin=-10)
    plt.show()

In [ ]:
geschummerte_darstellung(result_images_np[70])

In [ ]:
plt.imshow(common.CircleImage(11,11,5))

In [ ]:
from scipy.ndimage import convolve
from scipy.signal import fftconvolve


def filter_with_circle(input, circle_filter = None,radius=10):

    radius = int(radius)

    use_diameter = (2*radius) + 1

    if circle_filter is None:
        circle_filter = common.CircleImage(use_diameter,use_diameter,radius, inverted=True)


    use_pad = max(circle_filter.shape*2)
    
    input_padded = np.pad(input, use_pad, mode="constant", constant_values=0)
    gueltigkeit_padded = np.pad(np.ones_like(input), use_pad, mode="constant", constant_values=0)


    zaehler = fftconvolve(input_padded * gueltigkeit_padded, circle_filter, mode="same")
    nenner  = fftconvolve(gueltigkeit_padded, circle_filter, mode="same")

    ergebnis_padded = zaehler / np.maximum(nenner, 1e-6)
    ergebnis = ergebnis_padded[use_pad:-use_pad, use_pad:-use_pad]


    plt.imshow(ergebnis)
    return ergebnis

    

In [ ]:
import numpy as np
import numpy as np
from scipy.signal import fftconvolve


def convolve_images_with_masks(images, masks, nan_aware=True):
    """
    Faltet N Bilder mit M Masken.
    images : (N, H, W), darf NaN enthalten
    masks  : Liste von Masken, dürfen unterschiedlich groß sein
    return : (N, M, H, W)
    """
    N, H, W = images.shape
    M = len(masks)
    result = np.zeros((N, M, H, W), dtype=np.float32)

    if nan_aware:
        valid = (~np.isnan(images)).astype(np.float32)            # (N, H, W)
        clean = np.where(np.isnan(images), 0.0, images).astype(np.float32)
    else:
        clean = images.astype(np.float32)

    for i, mask in enumerate(masks):
        k = mask.astype(np.float32)[None, :, :]                   # (1, kh, kw)

        # mode="same" zentriert selbst, axes faltet pro Bild
        num = fftconvolve(clean, k, mode="same", axes=(-2, -1))   # (N, H, W)

        if nan_aware:
            den = fftconvolve(valid, k, mode="same", axes=(-2, -1))
            result[:, i] = num / np.maximum(den, 1e-6)
        else:
            result[:, i] = num

    return result

In [ ]:
fat_circle_masks_outer = common.CircleImageSeries(int(image_pixels.shape[0]*2.828),
                                                  int(image_pixels.shape[1]*2.828)
                                                  ,17,1,True)
fat_circle_masks_inner = common.CircleImageSeries(int(image_pixels.shape[0]*2.828),
                                                  int(image_pixels.shape[1]*2.828)
                                                  ,17,1,False)

In [ ]:
testimages_outer = convolve_images_with_masks(image_pixels[None,:,:,],fat_circle_masks_outer, nan_aware=True)
testimages_inner = convolve_images_with_masks(image_pixels[None,:,:,],fat_circle_masks_inner, nan_aware=True)

In [ ]:
plt.imshow(fat_circle_masks_inner[0])

In [ ]:
testimages_difference = testimages_inner[0] - testimages_outer[0]

In [ ]:
plt.imshow(testimages_difference[0])

In [ ]:
for img in testimages_difference:
    geschummerte_darstellung(img)

In [ ]:
plt.imshow(testimages_difference[16])

In [ ]:
cs = np.sum(testimages_difference, axis=0)

In [ ]:
plt.imshow(image_pixels)

In [ ]:
plt.imshow(cs)

In [ ]:
csb = np.roll(cs, -1, axis=0)
csb = np.roll(csb, -1, axis=1)
plt.imshow(csb-image_pixels)
np.std(csb-image_pixels)

In [ ]:
acht = np.linspace(8,8,4)

In [ ]:
np.pad(acht, 4, mode="constant", constant_values=0)[4:-4]

In [ ]:
def diameter_series(end, steps, finesteps=None):
    # "steps" Schritte inkl. 1 und "end"

    if finesteps is None: finesteps = steps

    factor = 10 **( (np.log10(end)) / (steps-1))

    series = np.logspace(0, np.log10(end), finesteps)

    return (series, factor)

diameter_series(300,5,16)

In [ ]:
finesteps = 30

diameters_first, factor_first = diameter_series(
    np.linalg.norm(image_pixels.shape,2) / 2, 17, finesteps
)
root_factor = np.sqrt(factor_first)
diameters_outer = diameters_first * root_factor
diameters_inner = diameters_first / root_factor


circle_masks_outer = []
circle_masks_inner = []

for i in range(finesteps):
    circle_masks_outer.append(
        common.CircleImage(
            image_pixels.shape[0]*2,
            image_pixels.shape[1]*2,
            diameters_outer[i],
            inverted=True,
        )
    )

    circle_masks_inner.append(
        common.CircleImage(
            image_pixels.shape[0]*2,
            image_pixels.shape[1]*2,
            diameters_inner[i],
            inverted=True,
        )
    )

pictures_outer = np.squeeze(
    convolve_images_with_masks(
        image_pixels[
            None,
            :,
            :,
        ],
        circle_masks_outer,
        nan_aware=True,
    ),
    axis=0,
)


pictures_inner =  np.squeeze(
    convolve_images_with_masks(
        image_pixels[
            None,
            :,
            :,
        ],
        circle_masks_inner,
        nan_aware=True,
    ),
    axis=0,
)



In [ ]:
pictures_combined_a = pictures_inner - pictures_outer

for pic in pictures_combined_a:
    plt.imshow(pic)
    plt.show()

In [ ]:
pictures_combined_b = pictures_inner - pictures_outer[-1] 

for pic in pictures_combined_b[::-1]:
    plt.imshow(pic)
    plt.show()

In [ ]:
pictures_combined_c =  pictures_inner[0] - pictures_outer[::-1] 

for pic in pictures_combined_c:
    plt.imshow(pic)
    plt.show()

In [ ]:
pictures_combined_all = np.concat((pictures_combined_a,pictures_combined_b[::-1],pictures_combined_c), axis = 0)
pictures_combined_all = pictures_combined_all[1:]

In [ ]:


pictures_combined_all_mean = np.mean(pictures_combined_all, axis=(1,2), keepdims=True)
pictures_combined_all_std = np.std(pictures_combined_all, axis=(1,2), keepdims=True)

pictures_combined_all_processed = pictures_combined_all - pictures_combined_all_mean
pictures_combined_all_processed = pictures_combined_all_processed / pictures_combined_all_std

pictures_combined_all_processed_mean = np.mean(pictures_combined_all_processed, axis=(1,2), keepdims=True)
pictures_combined_all_processed_std = np.std(pictures_combined_all_processed, axis=(1,2), keepdims=True)

pictures_combined_all_processed_mean_filtered = gaussian_filter(pictures_combined_all_processed_mean, mode="nearest", sigma=2)
pictures_combined_all_processed_std_filtered = gaussian_filter(pictures_combined_all_processed_std, mode="nearest", sigma=2)

pictures_combined_all_processed = pictures_combined_all_processed - pictures_combined_all_processed_mean_filtered
pictures_combined_all_processed = pictures_combined_all_processed / pictures_combined_all_processed_std_filtered

In [ ]:
for pic in pictures_combined_all_processed:
    plt.imshow(pic)
    plt.show()

In [ ]:
np.mean(pictures_combined_all_processed, axis=(1,2))

In [ ]:
pictures_combined_all_range = np.max(pictures_combined_all) - np.min(pictures_combined_all)
video_data = (pictures_combined_all - np.min(pictures_combined_all)) / pictures_combined_all_range

video_data = (video_data * 255 * 255).astype(np.uint16)

write_ffv1(video_data,"output/video.mkv",25)

## FFT Amplitude Profiles per Cluster

For a set of hardcoded representative positions (one per cluster), the nearest tiles
are identified and their median FFT amplitude profile is plotted as a bar chart.
This helps verify that the clusters correspond to visually distinct landscape types.

In [ ]:
representative_positions = {
    0: [(12.7796, 45.4373), (11.2246, 54.6149), (0.3724, 52.8735)],
    1: [(11.5235, 54.5613), (-0.0386, 52.5027), (-1.2785, 46.2522)],
    2: [(-1.0541, 53.8857), (11.9864, 52.7643), (13.5773, 53.8230)],
    3: [(3.1497, 50.3580), (-2.4971, 53.1608), (-0.4576, 53.2139)],
    4: [(10.7206, 45.2668), (-0.8134, 52.2503), (4.0064, 43.6073)],
    5: [(8.4562, 44.9047), (-7.2350, 53.9242), (-8.5240, 51.9537)],
    6: [(-2.2159, 42.4004), (-0.5915, 48.8978), (-9.0046, 52.2587)],
    7: [(-3.0215, 42.6649), (-8.9244, 52.0231), (8.5176, 44.5870)],
    8: [(11.5893, 44.0062), (8.9220, 44.5162), (-3.8236, 52.6587)],
    9: [(0.9308, 42.6556), (10.1733, 44.0430), (10.2515, 46.3518)],
}

from sklearn.metrics.pairwise import pairwise_distances

In [ ]:
representative_positions_np = np.array(list(representative_positions.values()))
representative_positions_np = representative_positions_np.reshape([representative_positions_np.shape[0]*representative_positions_np.shape[1],representative_positions_np.shape[2]])
representative_positions_np = np.flip(representative_positions_np, axis=1)
representative_positions_np.shape

In [ ]:
temporary_data["augmented_dems"][0].sealevel_mask

In [ ]:
# Koordinaten in der Form y, x!
all_coordinates = []
all_magnitudes = []
all_sealevels = []

for dem in temporary_data["augmented_dems"]:
    for tile_center in dem.tile_centers_orig:
        all_coordinates.append ((tile_center.y, tile_center.x))

    for magnitudes in dem.fft_magnitude_all_levels:
        all_magnitudes.append(magnitudes)

    for sealevels in dem.sealevel_mask:
        all_sealevels.append(sealevels)


all_coordinates_np = np.array(all_coordinates)
all_magnitudes_np = np.array(all_magnitudes)
all_sealevels_np = np.array(all_sealevels)

In [ ]:
all_coordinates_np.shape

In [ ]:
all_magnitudes_np.shape

In [ ]:
np.unique(all_sealevels_np, return_counts=True)

In [ ]:
filtered_magnitudes_np = all_magnitudes_np[all_sealevels_np]
filtered_coordinates_np = all_coordinates_np[all_sealevels_np]

In [ ]:
closest_amplitude_ids = pairwise_distances(representative_positions_np, filtered_coordinates_np).argsort(axis =1)

In [ ]:
closest_amplitude_ids.shape
closest_amplitude_ids_selection = closest_amplitude_ids[:,0:40]

In [ ]:
selected_amplitudes = filtered_magnitudes_np[closest_amplitude_ids_selection]

In [ ]:
import matplotlib.pyplot as plt

number_of_examples = closest_amplitudes.shape[0]
number_of_samples = closest_amplitudes.shape[1]

In [ ]:
median_amplitudes = np.median(selected_amplitudes, axis=1)

In [ ]:
ncols = 3
nrows = int(np.ceil(number_of_examples / ncols))

fig, ax = plt.subplots(
    nrows=nrows, ncols=ncols, sharex=True, sharey=True, figsize=(10, 10)
)

vmin, vmax = (median_amplitudes.min(), median_amplitudes.max())

for i in range(number_of_examples):
    row = i // ncols
    col = i % ncols
    heights = median_amplitudes[i]
    colors = plt.cm.viridis(1-((heights - vmin) / (vmax - vmin)))

    ax_temp = plt.Axes()

    ax[row, col].bar(np.array(range(number_of_samples))+1, median_amplitudes[i], color=colors)
    ax[row, col].boxplot(selected_amplitudes[i])
    ax[row, col].set_title(
        f"{representative_positions_np[i,0]:.2f}, {representative_positions_np[i,1]:.2f}"
    )
    ax[row, col].set_axisbelow(True)
    ax[row, col].grid(axis="x", visible=True, alpha=0.2, linestyle="--", which="major")
    ax[row, col].grid(axis="y", visible=True, alpha=0.9, linestyle="--", which="major")
    ax[row, col].set_yticks([0, 2, 4, 6, 8, 10])
    ax[row, col].set_yticklabels(["0","","4","","","10"])
    ax[row, col].set_xticks(np.array(range(number_of_samples + 1)) - 0.5)
    ax[row, col].set_xticklabels([],)
        


fig.tight_layout()
plt.show()

In [ ]:
import pickle
import io

# Figure in Buffer serialisieren
buf = io.BytesIO()
pickle.dump(fig, buf)

for i in range(30):
    # Figure aus Buffer kopieren
    buf.seek(0)
    fig_copy = pickle.load(buf)
    fig_copy.set_size_inches(10 * 3, 15 * 3)

    for ax_x in fig_copy.axes:
        ax_x.set_yticklabels([])
        ax_x.tick_params(left=False, bottom=False)
        ax_x.set_xlim([0.5, 17.5])
        ax_x.set_title("")
        for spine in ax_x.spines.values():
            spine.set_edgecolor("grey")
            spine.set_linewidth(0.5)

    # alle Axes außer der i-ten löschen
    for j, axis in enumerate(fig_copy.axes):
        if j != i:
            fig_copy.delaxes(axis)

    row = i // 3
    col = i % 3

    row += 1
    col += 1

    fig_copy.savefig(
        f"output/sample_fft_{row:02d}_{col:02d}.png", bbox_inches="tight", dpi=200
    )
    plt.close(fig_copy)

## Downloading DEMs for Representative Areas

Downloads small DEM tiles for each representative position so that example images
can be extracted and shown in documentation.
Originally developed as `download-dems-of-representative-areas.ipynb`.

In [ ]:
import os
import numpy as np
import rasterio
from rasterio.merge import merge
from rasterio.errors import RasterioIOError
from rasterio.warp import reproject, Resampling, calculate_default_transform
from rasterio.merge import merge as rio_merge
import pickle
from pyproj import CRS
import io


In [ ]:
with open('output/tabular_and_text/representative_positions.pkl', 'rb') as f:
    positions_ordered = pickle.load(f)

In [ ]:


# GDAL: HTTP-Range-Requests auf COGs beschleunigen
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"
os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = ".tif"


def km_to_degrees(km, lat):
    """Rechnet km in Grad um. Returns (delta_lon, delta_lat)."""
    km_per_degree_lat = 111.0
    km_per_degree_lon = 111.0 * np.cos(np.radians(lat))
    return km / km_per_degree_lon, km / km_per_degree_lat


def cop30_tile_url(lon_int, lat_int):
    """URL einer 1°×1°-COP30-Tile im AWS-Bucket von Sinergise."""
    ns = "N" if lat_int >= 0 else "S"
    ew = "E" if lon_int >= 0 else "W"
    name = (
        f"Copernicus_DSM_COG_10_"
        f"{ns}{abs(lat_int):02d}_00_"
        f"{ew}{abs(lon_int):03d}_00_DEM"
    )
    return f"https://copernicus-dem-30m.s3.amazonaws.com/{name}/{name}.tif"


def download_dem(lon, lat, extent_km, output_folder, label, idx):
    """Lädt ein DEM-GeoTIFF für eine Position."""
    delta_lon, delta_lat = km_to_degrees(extent_km / 2, lat)
    west, east = lon - delta_lon, lon + delta_lon
    south, north = lat - delta_lat, lat + delta_lat
    
    filename = f"{label}_{idx}_dem_{lon:.4f}_{lat:.4f}.tif"
    filepath = os.path.join(output_folder, filename)
    
    if os.path.exists(filepath):
        print(f"Überspringe {filename} (existiert bereits)")
        return
    
    # Welche 1°×1°-Tiles berührt die Bounding Box?
    lon_tiles = range(int(np.floor(west)), int(np.floor(east)) + 1)
    lat_tiles = range(int(np.floor(south)), int(np.floor(north)) + 1)
    
    print(f"Lade {filename}...")
    
    datasets = []
    for lon_int in lon_tiles:
        for lat_int in lat_tiles:
            url = cop30_tile_url(lon_int, lat_int)
            try:
                # /vsicurl/ holt per HTTP-Range-Request nur die nötigen Bytes
                datasets.append(rasterio.open(f"/vsicurl/{url}"))
            except RasterioIOError:
                # Tile fehlt (reiner Ozean, oder Land ohne öffentliche Freigabe)
                pass
    
    if not datasets:
        print(f"  ❌ Keine Tiles verfügbar für diesen Bereich")
        return
    
    # Tiles mergen und auf die gewünschte Bounding Box zuschneiden
    mosaic, out_transform = merge(datasets, bounds=(west, south, east, north))
    
    out_meta = datasets[0].meta.copy()
    out_meta.update({
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_transform,
        "compress": "deflate",
    })
    
    with rasterio.open(filepath, "w", **out_meta) as dst:
        dst.write(mosaic)
    
    for src in datasets:
        src.close()
    
    print(f"  Gespeichert: {filename}")


def download_representative_samples(
    positions_dict,
    extent_km=15,
    output_folder="Representative_Samples",
):
    """Lädt DEMs für alle Positionen."""
    os.makedirs(output_folder, exist_ok=True)
    
    for label, positions in positions_dict.items():
        for idx, position in enumerate(positions):
            lon, lat = float(position[0]), float(position[1])
            download_dem(lon, lat, extent_km, output_folder, label, idx)


def download_dem_aeqd(lon, lat, side_km, output_folder, label, idx):
    """
    Lädt ein DEM, projiziert es azimuthal-äquidistant um (lon, lat)
    und speichert einen quadratischen Ausschnitt der Größe side_km × side_km.
    """
    # 2× Puffer beim Download
    download_km = side_km * 2

    delta_lon, delta_lat = km_to_degrees(download_km / 2, lat)
    west, east = lon - delta_lon, lon + delta_lon
    south, north = lat - delta_lat, lat + delta_lat

    filename = f"{label}_{idx}_aeqd_{lon:.4f}_{lat:.4f}.tif"
    filepath = os.path.join(output_folder, filename)

    if os.path.exists(filepath):
        print(f"Überspringe {filename} (existiert bereits)")
        return

    # Tiles laden
    lon_tiles = range(int(np.floor(west)), int(np.floor(east)) + 1)
    lat_tiles = range(int(np.floor(south)), int(np.floor(north)) + 1)

    print(f"Lade {filename}...")

    datasets = []
    for lon_int in lon_tiles:
        for lat_int in lat_tiles:
            url = cop30_tile_url(lon_int, lat_int)
            try:
                datasets.append(rasterio.open(f"/vsicurl/{url}"))
            except RasterioIOError:
                pass

    if not datasets:
        print(f"  ❌ Keine Tiles verfügbar")
        return

    # Mosaic im WGS84-Bereich zusammensetzen
    mosaic, mosaic_transform = merge(datasets, bounds=(west, south, east, north))
    src_crs = datasets[0].crs
    for ds in datasets:
        ds.close()

    # Ziel-CRS: azimuthal äquidistant, zentriert auf (lon, lat)
    dst_crs = CRS.from_proj4(
        f"+proj=aeqd +lat_0={lat} +lon_0={lon} +datum=WGS84 +units=m"
    )

    print("Mosaic-Min:", np.min(mosaic))
    print("Mosaic-Max:", np.max(mosaic))
    print("Mosaic-Diff:", np.max(mosaic)-np.min(mosaic))

    # Auflösung aus dem Mosaic übernehmen (~30 m für COP30)
    res_m = mosaic_transform.a * 111_000  # grobe Umrechnung Grad → Meter

    # Ziel-Ausdehnung: nur der gewünschte Ausschnitt (kein Puffer mehr)
    half = (side_km * 1000) / 2
    dst_bounds = (-half, -half, half, half)  # (west, south, east, north) in Metern

    dst_transform, dst_width, dst_height = calculate_default_transform(
        src_crs,
        dst_crs,
        mosaic.shape[2],
        mosaic.shape[1],
        left=west, bottom=south, right=east, top=north,
        dst_width=int(side_km * 1000 / res_m),
        dst_height=int(side_km * 1000 / res_m),
    )

    # Transform manuell auf den Ausschnitt setzen
    pixel_size = side_km * 1000 / int(side_km * 1000 / res_m)
    dst_transform = rasterio.transform.from_bounds(
        *dst_bounds,
        width=int(side_km * 1000 / pixel_size),
        height=int(side_km * 1000 / pixel_size),
    )
    dst_width = dst_height = int(side_km * 1000 / pixel_size)


    dst_data = np.zeros((1, dst_height, dst_width), dtype=np.uint16)



    reproject(
        source=mosaic,
        destination=dst_data,
        src_transform=mosaic_transform,
        src_crs=src_crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.bilinear,
    )

    print("DST-Min:", np.min(dst_data))
    print("DST-Max:", np.max(dst_data))
    print("DST-Diff:", np.max(dst_data)-np.min(dst_data))

    
    out_meta = {
        "driver": "GTiff",
        "dtype": "uint16",
        "width": dst_width,
        "height": dst_height,
        "count": 1,
        "crs": dst_crs,
        "transform": dst_transform,
        "compress": "deflate",
    }

    with rasterio.open(filepath, "w", **out_meta) as dst:
        dst.write(dst_data)

    print(f"  Gespeichert: {filename} ({dst_width}×{dst_height} px)")



In [ ]:
documentation_italy_tile = (11.295510152613245, 42.910659637077316)

download_dem_aeqd(
    *documentation_italy_tile,
    12,
    "/Users/scharnagl/Documents/GitHub/geospatial-landscape-clustering-by-fft/documentation/Representative_Samples/16_bit_heightmaps",
    "italy",
    "sample_tile_16bit",
)

In [ ]:
error

In [ ]:

if __name__ == "__main__":
    positions = positions_ordered
    download_representative_samples(
        positions,
        extent_km=15,
        output_folder="documentation/Representative_Samples",
    )

In [ ]:
"""
Lädt Wasser-Masken (ESA WorldCover 10m v200, Jahr 2021) für die gleichen
Bereiche wie die DEMs aus dem OpenTopography-Skript.

- Quelle: öffentlicher S3-Bucket, kein API-Key, keine Rate Limits
- Auflösung nativ 10m, wird per Average-Resampling auf das DEM-Raster
  gebracht -> weiche Küsten (Anti-Aliasing)
- Ausgabe: 8-bit Graustufen-GeoTIFF (0 = Land, 255 = Wasser, Zwischenwerte
  an Rändern), gleiche Pixelgröße und exakt dasselbe Transform wie das DEM

Voraussetzung: pip install rasterio
"""



# /vsicurl/ lässt GDAL direkt per HTTPS aus dem COG lesen (Range Requests),
# ohne die ganze Kachel herunterzuladen.
WORLDCOVER_URL_TEMPLATE = (
    "/vsicurl/https://esa-worldcover.s3.eu-central-1.amazonaws.com/"
    "v200/2021/map/ESA_WorldCover_10m_2021_v200_{tile}_Map.tif"
)
WATER_CLASS = 80          # WorldCover-Code für "Permanent water bodies"
TILE_SIZE_DEG = 3         # WorldCover-Kacheln sind 3x3 Grad


def _tile_name(lat: int, lon: int) -> str:
    """Kachel-Name aus SW-Ecke. Beispiele: N48E011, S03W015."""
    lat_part = f"N{lat:02d}" if lat >= 0 else f"S{-lat:02d}"
    lon_part = f"E{lon:03d}" if lon >= 0 else f"W{-lon:03d}"
    return f"{lat_part}{lon_part}"


def _tile_urls_for_bbox(west, south, east, north):
    """Alle WorldCover-Kacheln, die die Bounding Box berühren."""
    lat0 = int(np.floor(south / TILE_SIZE_DEG)) * TILE_SIZE_DEG
    lat1 = int(np.floor((north - 1e-9) / TILE_SIZE_DEG)) * TILE_SIZE_DEG
    lon0 = int(np.floor(west / TILE_SIZE_DEG)) * TILE_SIZE_DEG
    lon1 = int(np.floor((east - 1e-9) / TILE_SIZE_DEG)) * TILE_SIZE_DEG

    urls = []
    for lat in range(lat0, lat1 + TILE_SIZE_DEG, TILE_SIZE_DEG):
        for lon in range(lon0, lon1 + TILE_SIZE_DEG, TILE_SIZE_DEG):
            urls.append(WORLDCOVER_URL_TEMPLATE.format(tile=_tile_name(lat, lon)))
    return urls


def download_water_mask(lon, lat, output_folder, label, idx):
    """Erzeugt die Wasser-Maske passend zur bereits vorhandenen DEM-Datei."""
    base       = f"{label}_{idx}_dem_{lon:.4f}_{lat:.4f}"
    dem_path   = os.path.join(output_folder, f"{base}.tif")
    water_path = os.path.join(output_folder, f"{base}_water.tif")

    if not os.path.exists(dem_path):
        print(f"⚠️  DEM fehlt, überspringe: {base}.tif")
        return
    if os.path.exists(water_path):
        print(f"Überspringe {base}_water.tif (existiert bereits)")
        return

    # Ziel-Raster 1:1 vom DEM übernehmen
    with rasterio.open(dem_path) as dem:
        dst_transform = dem.transform
        dst_crs       = dem.crs
        dst_height    = dem.height
        dst_width     = dem.width
        dst_bounds    = dem.bounds

    urls = _tile_urls_for_bbox(
        dst_bounds.left, dst_bounds.bottom, dst_bounds.right, dst_bounds.top
    )
    print(f"Lade {base}_water.tif ({len(urls)} Kachel(n))...")

    srcs = []
    try:
        for url in urls:
            try:
                srcs.append(rasterio.open(url))
            except RasterioIOError:
                # WorldCover liefert keine Kacheln für reine Ozean-Gebiete.
                # Fehlt eine Kachel, wird dieser Bereich später als Wasser gefüllt.
                print(f"  Keine Kachel (Ozean?): {os.path.basename(url)}")

        if not srcs:
            # Komplett außerhalb des Landes -> alles Wasser
            out = np.full((dst_height, dst_width), 255, dtype=np.uint8)
        else:
            # Kleiner Puffer, damit das Resampling keine Randpixel verliert
            pad = 0.002  # ~200 m
            mosaic, src_transform = rio_merge(
                srcs,
                bounds=(
                    dst_bounds.left   - pad,
                    dst_bounds.bottom - pad,
                    dst_bounds.right  + pad,
                    dst_bounds.top    + pad,
                ),
            )

            # Klassenraster -> binäre Maske als float (Wasser = 1.0, sonst 0.0).
            # Durch das Float-Resampling mit "average" entstehen weiche
            # Übergänge an den Küsten; erst ganz am Ende wird auf uint8 gecastet.
            water = (mosaic[0] == WATER_CLASS).astype(np.float32)

            dest = np.zeros((dst_height, dst_width), dtype=np.float32)
            reproject(
                source=water,
                destination=dest,
                src_transform=src_transform,
                src_crs="EPSG:4326",
                dst_transform=dst_transform,
                dst_crs=dst_crs,
                resampling=Resampling.average,
            )
            out = np.clip(dest * 255, 0, 255).astype(np.uint8)
    finally:
        for s in srcs:
            s.close()

    profile = {
        "driver":    "GTiff",
        "dtype":     "uint8",
        "count":     1,
        "width":     dst_width,
        "height":    dst_height,
        "crs":       dst_crs,
        "transform": dst_transform,
        "compress":  "deflate",
    }
    with rasterio.open(water_path, "w", **profile) as dst:
        dst.write(out, 1)
    print(f"  Gespeichert: {base}_water.tif")


def download_water_for_positions(positions_dict,
                                 output_folder="Representative_Samples"):
    """Läuft über dieselbe Struktur wie download_representative_samples."""
    os.makedirs(output_folder, exist_ok=True)
    for label, positions in positions_dict.items():
        for idx, position in enumerate(positions):
            lon, lat = float(position[0]), float(position[1])
            download_water_mask(lon, lat, output_folder, label, idx)


if __name__ == "__main__":
    # positions_ordered stammt aus deinem bestehenden Workflow
    positions = positions_ordered  # noqa: F821
    download_water_for_positions(
        positions,
        output_folder="Representative_Samples",
    )

## Extracting Sample Images for Documentation

Crops and exports small images from the labeled output GeoTIFFs.
Originally developed as `extract_sample_images.ipynb`.

In [ ]:
from rasterio.warp import transform_bounds, reproject, Resampling
from rasterio.transform import from_origin
import numpy as np
import rasterio


SHAPE_OF_INTEREST = [
    {
        "type": "Polygon",
        "coordinates": [[((5, 46)), (5, 36), (18.5, 36), (18.5, 46)]],
    }
]

x_center = 0
y_center = 0

for pair in SHAPE_OF_INTEREST[0]["coordinates"][0]:
    x_center += pair[0] / 4
    y_center += pair[1] / 4


dst_crs = f"+proj=aeqd +lat_0={y_center} +lon_0={x_center} +x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"


def build_fixed_grid(resolution_m):
    """Zielgitter nur aus dem Polygon und der Auflösung – unabhängig vom Input."""
    coords = SHAPE_OF_INTEREST[0]["coordinates"][0]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]

    # Polygon-Bounds (Grad) in die Ziel-CRS (Meter) umrechnen
    left, bottom, right, top = transform_bounds(
        "EPSG:4326",
        dst_crs,
        min(lons),
        min(lats),
        max(lons),
        max(lats),
        densify_pts=21,
    )

    # Auf die Auflösung einrasten -> Ursprung sitzt bei jeder Datei gleich
    left = np.floor(left / resolution_m) * resolution_m
    bottom = np.floor(bottom / resolution_m) * resolution_m
    right = np.ceil(right / resolution_m) * resolution_m
    top = np.ceil(top / resolution_m) * resolution_m

    width = int((right - left) / resolution_m)
    height = int((top - bottom) / resolution_m)
    transform = from_origin(left, top, resolution_m, resolution_m)
    return transform, width, height


def export_area(input_file: str, output_file: str, isdem, resolution_m=1000):
    transform, width, height = build_fixed_grid(resolution_m)

    with rasterio.open(input_file) as src:
        out_meta = src.meta.copy()
        out_meta.update(
            {
                "crs": dst_crs,
                "transform": transform,
                "width": width,
                "height": height,
                "count": src.count,
                "dtype": "float16",
            }
        )

        # Jede Quelle in genau dieses feste Gitter projizieren (als float32)
        # Nicht abgedeckte Pixel bleiben NaN, damit sie die Streckung nicht verfälschen
        projected = np.full((src.count, height, width), np.nan, dtype="float32")
        for i in range(src.count):
            reproject(
                source=rasterio.band(src, i + 1),
                destination=projected[i],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                dst_nodata=np.nan,
                resampling=Resampling.bilinear if isdem else Resampling.nearest,
            )

        with rasterio.open(output_file, "w", **out_meta) as destination:
            for i in range(src.count):
                if isdem:
                    band = np.clip(projected[i], 0, None)  # Negatives -> 0 (wie gehabt)
                    vmin = np.nanmin(band)
                    vmax = np.nanmax(band)
                    scaled = band / (vmax - vmin) * 255 *255 # deine Formel
                    out = np.nan_to_num(scaled, nan=0).astype("float16")
                else:
                    out = (
                        np.nan_to_num(projected[i], nan=0).clip(0, 255).astype("uint8")
                    )

                destination.write(out, i + 1)

In [ ]:


export_area('/Users/scharnagl/Documents/GitHub/geospatial-landscape-clustering-by-fft/input_geotiffs/geotiff 002.2, 035.2, 020.4, 050.3.tif', "output/export.tif", True)


In [ ]:


export_area("/Users/scharnagl/Documents/GitHub/geospatial-landscape-clustering-by-fft/output/label_images/geotiff 002.2, 035.2, 020.4, 050.3.tif 277c  tlszkm 12.0  tlszpx 23  fftlvls 17  6.7x -ovrlp-pct 85 fltrrds 15 lblct 10.tif", "output/export_c_X.tif", False)

In [ ]:


export_area("/Users/scharnagl/Documents/GitHub/geospatial-landscape-clustering-by-fft/output/label_images/geotiff 002.2, 035.2, 020.4, 050.3.tif b5eb  tlszkm 12.0  tlszpx 23  fftlvls 17  6.7x -ovrlp-pct 85 fltrrds 15 lblct 10.tif", "output/export_b_X.tif", False)

## Reprojecting Representative Samples

Reprojects the downloaded representative DEM tiles from geographic CRS to metric projection.
Originally developed as `tertiary_download_representative_samples.ipynb`.

In [ ]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.crs import CRS
import numpy as np
import glob
import os

def convert_to_16bit_heightmaps(
    input_folder="Representative_Samples",
    output_folder="16_bit_heightmaps",
    min_height=-100,
    max_height=10000
):
    """
    Wandelt DEMs in 16-bit Heightmaps um mit orthographischer Projektion.
    
    min_height: Minimale Höhe in Metern (wird zu 0)
    max_height: Maximale Höhe in Metern (wird zu 65535)
    """
    # Ausgabeordner erstellen
    output_path = os.path.join(input_folder, output_folder)
    os.makedirs(output_path, exist_ok=True)
    
    # Alle TIF-Dateien finden
    tif_files = glob.glob(os.path.join(input_folder, "*.tif"))
    
    if not tif_files:
        print(f"Keine TIF-Dateien in {input_folder} gefunden")
        return
    
    print(f"Konvertiere {len(tif_files)} Dateien...")
    
    for tif_file in tif_files:
        filename = os.path.basename(tif_file)
        output_file = os.path.join(output_path, filename)
        
        # Prüfen ob bereits existiert
        if os.path.exists(output_file):
            print(f"Überspringe {filename} (existiert bereits)")
            continue
        
        # DEM öffnen
        with rasterio.open(tif_file) as src:
            # Mittelpunkt des DEMs berechnen
            bounds = src.bounds
            center_lon = (bounds.left + bounds.right) / 2
            center_lat = (bounds.bottom + bounds.top) / 2
            
            # Orthographische Projektion um Mittelpunkt definieren
            ortho_crs = CRS.from_proj4(
                f"+proj=ortho +lat_0={center_lat} +lon_0={center_lon} "
                f"+x_0=0 +y_0=0 +ellps=WGS84 +units=m +no_defs"
            )
            
            # Transform und Dimensionen für Reprojection berechnen
            transform, width, height = calculate_default_transform(
                src.crs, ortho_crs, src.width, src.height, *bounds
            )
            
            # Neues Array für reprojizierte Daten
            dem_reprojected = np.empty((height, width), dtype=np.float32)
            
            # Reprojizieren
            reproject(
                source=rasterio.band(src, 1),
                destination=dem_reprojected,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=ortho_crs,
                resampling=Resampling.bilinear
            )
            
            # Auf -100 bis 10000m clippen
            dem_clipped = np.clip(dem_reprojected, min_height, max_height)
            
            # Auf 0-65535 normalisieren
            normalized = (dem_clipped - min_height) / (max_height - min_height)
            heightmap_16bit = (normalized * 65535).astype(np.uint16)
            
            # Als 16-bit TIFF speichern
            profile = {
                'driver': 'GTiff',
                'dtype': rasterio.uint16,
                'width': width,
                'height': height,
                'count': 1,
                'crs': ortho_crs,
                'transform': transform,
                'compress': 'lzw'
            }
            
            with rasterio.open(output_file, 'w', **profile) as dst:
                dst.write(heightmap_16bit, 1)
        
        print(f"  Konvertiert: {filename}")
    
    print(f"Fertig. Dateien in: {output_path}")
    

# Beispiel-Aufruf:
if __name__ == "__main__":
    convert_to_16bit_heightmaps()

## Geographic Descriptions of Representative Positions

Resolves representative geographic coordinates to human-readable place names
using the `geonamescache` library.
Originally developed as `get-representative-descriptions.ipynb`.

In [ ]:
import pickle
with open('positions_2026-04-23.pkl', 'rb') as f:
    positions_ordered = pickle.load(f)


In [ ]:
import numpy as np
import pandas as pd
import geonamescache

# --- KONFIGURATION ---
CITY_MIN_POPULATION = 150000

# --- EINGABE ---
clusters =positions_ordered

# --- HILFSFUNKTIONEN ---

def haversine_np(lon1, lat1, lon2, lat2):
    """Entfernung in km (Eingabe: Radians)."""
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))


def bearing_deg(lat1, lon1, lat2, lon2):
    """Kurswinkel in Grad (0° = Nord, im Uhrzeigersinn)."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360


def bearing_to_direction(deg):
    """Kurswinkel → Himmelsrichtung (Achtelkreise)."""
    directions = ["N", "NO", "O", "SO", "S", "SW", "W", "NW"]
    idx = round((deg-180) / 45) % 8
    return directions[idx]


def round_half(value):
    """Rundet auf 0,5 km."""
    return round(value * 2) / 2


def get_nearest_city(point_lat, point_lon, city_df, city_coords_rad):
    """Nächste Stadt + Entfernung in km."""
    p_lat = np.radians(point_lat)
    p_lon = np.radians(point_lon)
    dists = haversine_np(p_lon, p_lat, city_coords_rad[:, 1], city_coords_rad[:, 0])
    idx = np.argmin(dists)
    city = city_df.iloc[idx]
    return city["name"], city["lat"], city["lon"], city["country"], dists[idx]


def format_label(dist_km, direction, city_name):
    """Erstellt z.B. '15,5 km nördlich von Calais'."""
    direction_text = {
        "N": "nördlich",
        "NO": "nordöstlich",
        "O": "östlich",
        "SO": "südöstlich",
        "S": "südlich",
        "SW": "südwestlich",
        "W": "westlich",
        "NW": "nordwestlich",
    }
    dist_str = f"{dist_km:g}".replace(".", ",")
    if dist_km == 0.0:
        return city_name
    return f"{dist_str} km {direction_text[direction]} von {city_name}"


# --- STÄDTE-DATENBANK ---
gc = geonamescache.GeonamesCache()
cities = gc.get_cities()
countries_raw = gc.get_countries()

# Ländernamen auf Deutsch (nur europäische Länder, die hier vorkommen können)
COUNTRY_DE = {
    "AD": "Andorra", "AL": "Albanien", "AT": "Österreich", "BA": "Bosnien-Herzegowina",
    "BE": "Belgien", "BG": "Bulgarien", "BY": "Weißrussland", "CH": "Schweiz",
    "CY": "Zypern", "CZ": "Tschechien", "DE": "Deutschland", "DK": "Dänemark",
    "EE": "Estland", "ES": "Spanien", "FI": "Finnland", "FR": "Frankreich",
    "GB": "Vereinigtes Königreich", "GR": "Griechenland", "HR": "Kroatien",
    "HU": "Ungarn", "IE": "Irland", "IS": "Island", "IT": "Italien",
    "LI": "Liechtenstein", "LT": "Litauen", "LU": "Luxemburg", "LV": "Lettland",
    "MC": "Monaco", "MD": "Moldau", "ME": "Montenegro", "MK": "Nordmazedonien",
    "MT": "Malta", "NL": "Niederlande", "NO": "Norwegen", "PL": "Polen",
    "PT": "Portugal", "RO": "Rumänien", "RS": "Serbien", "RU": "Russland",
    "SE": "Schweden", "SI": "Slowenien", "SK": "Slowakei", "SM": "San Marino",
    "TR": "Türkei", "UA": "Ukraine", "VA": "Vatikanstadt", "XK": "Kosovo",
}

european_cities = [
    {
        "name": c["name"],
        "lat": c["latitude"],
        "lon": c["longitude"],
        "country": COUNTRY_DE.get(c["countrycode"], countries_raw.get(c["countrycode"], {}).get("name", c["countrycode"])),
    }
    for c in cities.values()
    if c["timezone"] and c["timezone"].startswith("Europe/")
    and c["population"] > CITY_MIN_POPULATION
]
city_df = pd.DataFrame(european_cities)
city_coords_rad = np.radians(city_df[["lat", "lon"]].values)

# --- HAUPTSCHLEIFE ---
rows = []

for group_id, positions in clusters.items():
    for pos_idx, (lon, lat) in enumerate(positions):
        city_name, city_lat, city_lon, country, dist_km = get_nearest_city(
            lat, lon, city_df, city_coords_rad
        )
        dist_rounded = round_half(dist_km)
        deg = bearing_deg(lat, lon, city_lat, city_lon)
        direction = bearing_to_direction(deg)
        label = format_label(dist_rounded, direction, city_name) + f", {country}"

        rows.append({
            "gruppe":          group_id,
            "nr":              pos_idx,
            "lon":             round(lon, 6),
            "lat":             round(lat, 6),
            "stadt":           city_name,
            "land":            country,
            "entfernung_km":   dist_rounded,
            "richtung":        direction,
            "label":           label,
        })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

df.to_csv("output/tabular_and_text/cluster_locations.csv", index=False)
print("\nGespeichert als cluster_locations.csv")